In [ ]:
# FUNGVA_model_definition.py

import torch
import torch.nn as nn
import torch.nn.functional as F


##############################################################
############## Building the Modules for the model ############
##############################################################

class GraphCNN(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()

        self.linear = nn.Linear(
            in_features,
            out_features,
            bias=bias,
        )

        self.mask = None

    def set_mask(self, mask):
        """
        Equivalent to the R method:

            self$mask <- mask$detach()$to(
              dtype  = self$linear$weight$dtype,
              device = self$linear$weight$device
            )
            self$mask$requires_grad_(FALSE)

        The mask should have the same shape as self.linear.weight:
            out_features x in_features
        """
        self.mask = mask.detach().to(
            dtype=self.linear.weight.dtype,
            device=self.linear.weight.device,
        )
        self.mask.requires_grad_(False)

    def forward(self, x):
        if self.mask is not None:
            w = self.linear.weight * self.mask
            return F.linear(x, w, self.linear.bias)

        return self.linear(x)


class FUNGVA(nn.Module):
    def __init__(self, latent_dim, cov_dim):
        super().__init__()

        self.latent_dim = latent_dim
        self.cov_dim = cov_dim

        p = (68 * 67) // 2

        # R equivalent:
        # idx <- torch_tril(torch_ones(68, 68, dtype = torch_bool()), diagonal = -1)$nonzero()
        # lt_flat_idx <- ((idx[, 1] - 1) * 68 + idx[, 2])$to(dtype = torch_long())
        #
        # In Python/PyTorch, nonzero() returns 0-based indices directly.
        idx = torch.tril(
            torch.ones(68, 68, dtype=torch.bool),
            diagonal=-1,
        ).nonzero(as_tuple=False)

        lt_flat_idx = idx[:, 0] * 68 + idx[:, 1]
        self.register_buffer("lt_flat_idx", lt_flat_idx.long())

        # Encoder
        self.en_mu_la1 = nn.Linear(p, 1024, bias=False)
        self.en_var_la1 = nn.Linear(p, 1024, bias=False)

        self.en_mu_la2 = nn.Linear(1024, 128, bias=False)
        self.en_var_la2 = nn.Linear(1024, 128, bias=False)

        self.en_mu_la3 = nn.Linear(128, latent_dim)
        self.en_var_la3 = nn.Linear(128, latent_dim)

        # Decoder heads
        self.dc1_la1 = nn.Linear(latent_dim, 68)
        self.dc2_la1 = nn.Linear(latent_dim, 68)
        self.dc3_la1 = nn.Linear(latent_dim, 68)
        self.dc4_la1 = nn.Linear(latent_dim, 68)
        self.dc5_la1 = nn.Linear(latent_dim, 68)

        self.dc1_la2 = GraphCNN(68, 68)
        self.dc2_la2 = GraphCNN(68, 68)
        self.dc3_la2 = GraphCNN(68, 68)
        self.dc4_la2 = GraphCNN(68, 68)
        self.dc5_la2 = GraphCNN(68, 68)

        self.dcintercept = GraphCNN(p, p)

        # FiLM nets
        film_hidden = 32

        self.film1 = nn.Sequential(
            nn.Linear(cov_dim, film_hidden),
            nn.ReLU(),
            nn.Linear(film_hidden, 2 * 68),
        )

        self.film2 = nn.Sequential(
            nn.Linear(cov_dim, film_hidden),
            nn.ReLU(),
            nn.Linear(film_hidden, 2 * 68),
        )

        self.film3 = nn.Sequential(
            nn.Linear(cov_dim, film_hidden),
            nn.ReLU(),
            nn.Linear(film_hidden, 2 * 68),
        )

        self.film4 = nn.Sequential(
            nn.Linear(cov_dim, film_hidden),
            nn.ReLU(),
            nn.Linear(film_hidden, 2 * 68),
        )

        self.film5 = nn.Sequential(
            nn.Linear(cov_dim, film_hidden),
            nn.ReLU(),
            nn.Linear(film_hidden, 2 * 68),
        )

        self.tanh_act = nn.Tanh()

    def set_mask(self, masks):
        """
        Equivalent to:

            self$dc1_la2$set_mask(masks[[1]])
            self$dc2_la2$set_mask(masks[[2]])
            self$dc3_la2$set_mask(masks[[3]])
            self$dc4_la2$set_mask(masks[[4]])
            self$dc5_la2$set_mask(masks[[5]])
            self$dcintercept$set_mask(masks[[6]])

        In Python, masks should be a list/tuple of length 6.
        """
        self.dc1_la2.set_mask(masks[0])
        self.dc2_la2.set_mask(masks[1])
        self.dc3_la2.set_mask(masks[2])
        self.dc4_la2.set_mask(masks[3])
        self.dc5_la2.set_mask(masks[4])
        self.dcintercept.set_mask(masks[5])

    def encode(self, x):
        mu = self.en_mu_la1(x)
        mu = F.relu(mu)
        mu = self.en_mu_la2(mu)
        mu = F.relu(mu)
        mu = self.en_mu_la3(mu)

        logvar = self.en_var_la1(x)
        logvar = F.relu(logvar)
        logvar = self.en_var_la2(logvar)
        logvar = F.relu(logvar)
        logvar = self.en_var_la3(logvar)

        return {
            "mu": mu,
            "logvar": logvar,
        }

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        return mu + eps * std

    def film_modulate(self, h, cov, film_net):
        film_params = film_net(cov)

        # R:
        # gamma <- film_params$narrow(2, 1, 68)
        # beta  <- film_params$narrow(2, 69, 68)
        #
        # Python uses 0-based indexing.
        gamma = film_params[:, :68]
        beta = film_params[:, 68:136]

        return (1 + gamma) * h + beta

    def _decode_head(self, z, cov, dc_la1, dc_la2, film_net):
        lt_idx = self.lt_flat_idx

        dcl_pre = dc_la1(z)
        dcl_pre = F.relu(dcl_pre)

        dcl_mod = self.film_modulate(dcl_pre, cov, film_net)

        dcl = dc_la2(dcl_mod)

        # R:
        # dcl_out <- torch_bmm(dcl$unsqueeze(3), dcl$unsqueeze(2))
        #
        # R torch dimensions are 1-based. For a batch x 68 tensor:
        #   unsqueeze(3) -> batch x 68 x 1
        #   unsqueeze(2) -> batch x 1 x 68
        #
        # Python:
        dcl_out = torch.bmm(
            dcl.unsqueeze(2),
            dcl.unsqueeze(1),
        )

        dcl_vec = dcl_out.reshape(dcl_out.shape[0], 68 * 68)
        dcl_vec = dcl_vec.index_select(dim=1, index=lt_idx)

        return dcl_vec

    def decode(self, z, cov):
        dcl1_vec = self._decode_head(
            z=z,
            cov=cov,
            dc_la1=self.dc1_la1,
            dc_la2=self.dc1_la2,
            film_net=self.film1,
        )

        dcl2_vec = self._decode_head(
            z=z,
            cov=cov,
            dc_la1=self.dc2_la1,
            dc_la2=self.dc2_la2,
            film_net=self.film2,
        )

        dcl3_vec = self._decode_head(
            z=z,
            cov=cov,
            dc_la1=self.dc3_la1,
            dc_la2=self.dc3_la2,
            film_net=self.film3,
        )

        dcl4_vec = self._decode_head(
            z=z,
            cov=cov,
            dc_la1=self.dc4_la1,
            dc_la2=self.dc4_la2,
            film_net=self.film4,
        )

        dcl5_vec = self._decode_head(
            z=z,
            cov=cov,
            dc_la1=self.dc5_la1,
            dc_la2=self.dc5_la2,
            film_net=self.film5,
        )

        dcl_out = dcl1_vec + dcl2_vec + dcl3_vec + dcl4_vec + dcl5_vec

        dc_output = self.dcintercept(dcl_out)
        fc_pred = self.tanh_act(dc_output)

        return {
            "recon": dc_output,
            "fc_pred": fc_pred,
        }

    def forward(self, x):
        """
        Expects x to match the dataset batch structure:

            x = {
                "fc": fc_tensor,
                "cov": cov_tensor,
            }

        where:
            fc_tensor  has shape batch_size x 2278
            cov_tensor has shape batch_size x cov_dim
        """
        fc = x["fc"]
        cov = x["cov"]

        enc = self.encode(fc)

        mu = enc["mu"]
        logvar = enc["logvar"]

        z = self.reparameterize(mu, logvar)
        dec = self.decode(z, cov)

        return {
            "recon": dec["recon"],
            "mu": mu,
            "logvar": logvar,
            "fc_pred": dec["fc_pred"],
            "z": z,
            "cov": cov,
        }


class FUNGVA_Trainer(nn.Module):
    def __init__(self, latent_dim, cov_dim, masks=None):
        super().__init__()

        self.model = FUNGVA(
            latent_dim=latent_dim,
            cov_dim=cov_dim,
        )

        if masks is not None:
            self.model.set_mask(masks)

    def forward(self, x):
        return self.model(x)